<a href="https://colab.research.google.com/github/najmunoishi/Baust_Student_Communication/blob/master/ai_academic_assistant_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q sentence-transformers faiss-cpu numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.1 MB/s eta 0:00:00


academic dataset

In [2]:
import json
import os

os.makedirs("data", exist_ok=True)
os.makedirs("tests", exist_ok=True)

documents = [
    {
        "source_id": "DOC001",
        "title": "Course Prerequisites",
        "category": "registration",
        "text": (
            "Students should check the course catalogue to identify "
            "prerequisites before registering. If a prerequisite is "
            "unclear, contact the academic adviser or department."
        )
    },
    {
        "source_id": "DOC002",
        "title": "Course Withdrawal",
        "category": "registration",
        "text": (
            "Students considering course withdrawal should review "
            "the academic calendar, applicable deadlines, and withdrawal "
            "rules. Contact the academic office if the rules are unclear."
        )
    },
    {
        "source_id": "DOC003",
        "title": "Academic Advising",
        "category": "advising",
        "text": (
            "Students preparing for academic advising should list their "
            "programme, completed courses, current courses, and questions "
            "about degree progression. Avoid sharing passwords or "
            "unnecessary sensitive personal information."
        )
    },
    {
        "source_id": "DOC004",
        "title": "Examination Schedule",
        "category": "exams",
        "text": (
            "Students should review the official examination schedule, "
            "course syllabus, and instructor announcements. Report schedule "
            "conflicts to the relevant academic office."
        )
    },
    {
        "source_id": "DOC005",
        "title": "Student Data Privacy",
        "category": "privacy",
        "text": (
            "An academic-support chatbot should avoid collecting "
            "unnecessary personally identifiable information. Sensitive "
            "data should be redacted before document ingestion or storage."
        )
    },
    {
        "source_id": "DOC006",
        "title": "Contacting an Academic Office",
        "category": "support",
        "text": (
            "When contacting an academic office, use a clear subject line, "
            "explain the question concisely, and include only information "
            "necessary to resolve the issue. Verify important deadlines "
            "against current official documents."
        )
    }
]

with open("data/academic_faqs.jsonl", "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

print("Total documents:", len(documents))
print("First document:", documents[0])

Total documents: 6
First document: {'source_id': 'DOC001', 'title': 'Course Prerequisites', 'category': 'registration', 'text': 'Students should check the course catalogue to identify prerequisites before registering. If a prerequisite is unclear, contact the academic adviser or department.'}


Text embedding and FAISS index

In [3]:
import json
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


class AcademicRAG:

    def __init__(
        self,
        data_path="data/academic_faqs.jsonl"
    ):
        self.data_path = Path(data_path)
        self.documents = self.load_documents()

        if not self.documents:
            raise ValueError("Knowledge base is empty.")

        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

        texts = [
            f"{doc['title']}. {doc['text']}"
            for doc in self.documents
        ]

        embeddings = self.model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        embeddings = np.asarray(
            embeddings,
            dtype="float32"
        )

        dimension = embeddings.shape[1]

        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(embeddings)

    def load_documents(self):
        records = []

        with self.data_path.open(
            "r", encoding="utf-8"
        ) as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line))

        return records

    def retrieve(self, question, top_k=3):
        query_embedding = self.model.encode(
            [question],
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        k = min(top_k, len(self.documents))

        scores, indices = self.index.search(
            query_embedding, k
        )

        results = []

        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue

            doc = self.documents[int(idx)]

            results.append({
                "source_id": doc["source_id"],
                "title": doc["title"],
                "category": doc["category"],
                "text": doc["text"],
                "score": float(score)
            })

        return results

    def answer(self, question, top_k=3):
        sources = self.retrieve(question, top_k)

        if not sources:
            return {
                "answer": "No relevant documents found.",
                "sources": []
            }

        evidence = "\n\n".join(
            f"- {doc['text']} "
            f"(Source: {doc['source_id']})"
            for doc in sources
        )

        answer = (
            "Relevant information from the knowledge base:\n\n"
            + evidence
            + "\n\nPlease verify important rules against "
              "current official documents."
        )

        return {
            "answer": answer,
            "sources": sources
        }


print("RAG engine class defined successfully.")

RAG engine class defined successfully.


model index create

In [4]:
assistant = AcademicRAG()

print("Documents indexed:", len(assistant.documents))
print("FAISS vectors:", assistant.index.ntotal)
print("Embedding dimension:", assistant.index.d)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents indexed: 6
FAISS vectors: 6
Embedding dimension: 384


In [5]:
question = "How can I check course prerequisites?"

result = assistant.answer(question, top_k=3)

print("QUESTION:", question)
print("\nANSWER:\n")
print(result["answer"])

print("\nRETRIEVED SOURCES:")
for source in result["sources"]:
    print(
        source["source_id"],
        "|",
        source["title"],
        "| score:",
        round(source["score"], 3)
    )

QUESTION: How can I check course prerequisites?

ANSWER:

Relevant information from the knowledge base:

- Students should check the course catalogue to identify prerequisites before registering. If a prerequisite is unclear, contact the academic adviser or department. (Source: DOC001)

- Students considering course withdrawal should review the academic calendar, applicable deadlines, and withdrawal rules. Contact the academic office if the rules are unclear. (Source: DOC002)

- Students should review the official examination schedule, course syllabus, and instructor announcements. Report schedule conflicts to the relevant academic office. (Source: DOC004)

Please verify important rules against current official documents.

RETRIEVED SOURCES:
DOC001 | Course Prerequisites | score: 0.801
DOC002 | Course Withdrawal | score: 0.405
DOC004 | Examination Schedule | score: 0.403


In [6]:
question = "How should a chatbot protect student privacy?"

result = assistant.answer(question, top_k=3)

print(result["answer"])

Relevant information from the knowledge base:

- An academic-support chatbot should avoid collecting unnecessary personally identifiable information. Sensitive data should be redacted before document ingestion or storage. (Source: DOC005)

- Students preparing for academic advising should list their programme, completed courses, current courses, and questions about degree progression. Avoid sharing passwords or unnecessary sensitive personal information. (Source: DOC003)

- Students should review the official examination schedule, course syllabus, and instructor announcements. Report schedule conflicts to the relevant academic office. (Source: DOC004)

Please verify important rules against current official documents.


retrieval evaluation

In [7]:
test_cases = [
    {
        "question": "How can I check course prerequisites?",
        "expected_source": "DOC001"
    },
    {
        "question": "What should I do before withdrawing?",
        "expected_source": "DOC002"
    },
    {
        "question": "What should I prepare for academic advising?",
        "expected_source": "DOC003"
    },
    {
        "question": "How can a chatbot protect personal data?",
        "expected_source": "DOC005"
    },
    {
        "question": "How do I contact the academic office?",
        "expected_source": "DOC006"
    }
]

evaluation_rows = []

for case in test_cases:
    retrieved = assistant.retrieve(
        case["question"],
        top_k=3
    )

    retrieved_ids = [
        item["source_id"] for item in retrieved
    ]

    evaluation_rows.append({
        "question": case["question"],
        "expected_source": case["expected_source"],
        "retrieved_sources": retrieved_ids,
        "hit_at_3": int(
            case["expected_source"] in retrieved_ids
        )
    })

import pandas as pd

evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)

hit_rate = evaluation_df["hit_at_3"].mean()

print(f"Hit@3: {hit_rate:.2%}")

,question,expected_source,retrieved_sources,hit_at_3
0,How can I check course prerequisites?,DOC001,"[DOC001, DOC002, DOC004]",1
1,What should I do before withdrawing?,DOC002,"[DOC002, DOC004, DOC006]",1
2,What should I prepare for academic advising?,DOC003,"[DOC003, DOC006, DOC001]",1
3,How can a chatbot protect personal data?,DOC005,"[DOC005, DOC003, DOC006]",1
4,How do I contact the academic office?,DOC006,"[DOC006, DOC004, DOC002]",1


Hit@3: 100.00%


evaluation result

In [8]:
evaluation_df.to_csv(
    "evaluation_results.csv",
    index=False
)

print("Evaluation results saved.")

Evaluation results saved.


source aware chatbot function

In [9]:
def chat():
    print("Academic Assistant")
    print("Type 'exit' to stop.\n")

    while True:
        question = input("You: ").strip()

        if question.lower() == "exit":
            print("Session ended.")
            break

        if not question:
            print("Please enter a question.")
            continue

        result = assistant.answer(question, top_k=3)

        print("\nAssistant:")
        print(result["answer"])

        print("\nSources:")
        for source in result["sources"]:
            print(
                f"- {source['source_id']}: "
                f"{source['title']}"
            )

        print("\n" + "-" * 50 + "\n")


chat()

Academic Assistant
Type 'exit' to stop.

You: exit
Session ended.


In [11]:
from pathlib import Path
import json
import textwrap
import zipfile

project_dir = Path("/content/ai-academic-assistant-rag")
(project_dir / "data").mkdir(parents=True, exist_ok=True)
(project_dir / "tests").mkdir(parents=True, exist_ok=True)

# Save the dataset
with (project_dir / "data" / "academic_faqs.jsonl").open(
    "w", encoding="utf-8"
) as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

# Save the RAG engine
rag_code = '''
import json
from pathlib import Path
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


class AcademicRAG:
    def __init__(self, data_path="data/academic_faqs.jsonl"):
        self.data_path = Path(data_path)
        self.documents = self.load_documents()

        if not self.documents:
            raise ValueError("Knowledge base is empty.")

        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

        texts = [
            f"{doc['title']}. {doc['text']}"
            for doc in self.documents
        ]

        vectors = self.model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        self.index = faiss.IndexFlatIP(vectors.shape[1])
        self.index.add(vectors)

    def load_documents(self):
        with self.data_path.open(encoding="utf-8") as f:
            return [
                json.loads(line)
                for line in f if line.strip()
            ]

    def retrieve(self, question, top_k=3):
        query_vector = self.model.encode(
            [question],
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

        k = min(top_k, len(self.documents))
        scores, indices = self.index.search(query_vector, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            doc = self.documents[int(idx)]
            results.append({
                **doc,
                "score": float(score)
            })
        return results

    def answer(self, question, top_k=3):
        sources = self.retrieve(question, top_k)

        if not sources:
            return {"answer": "No relevant documents found.",
                    "sources": []}

        evidence = "\\\\n\\\\n".join(
            f"- {doc['text']} (Source: {doc['source_id']})"
            for doc in sources
        )

        answer = (
            "Relevant information from the knowledge base:\\\\n\\\\n"
            + evidence
            + "\\\\n\\\\nVerify important rules against current official documents."
        )

        return {"answer": answer, "sources": sources}
'''

(project_dir / "rag_engine.py").write_text(
    textwrap.dedent(rag_code).lstrip(),
    encoding="utf-8"
)

# Save Streamlit app
app_code = '''
import streamlit as st
from rag_engine import AcademicRAG

st.set_page_config(page_title="AI Academic Assistant", page_icon="🎓")
st.title("AI Academic Assistant")
st.caption("RAG + semantic retrieval + source references")

@st.cache_resource
def load_assistant():
    return AcademicRAG("data/academic_faqs.jsonl")

st.info("Demo data only. Verify important rules using official sources.")
assistant = load_assistant()

top_k = st.sidebar.slider("Retrieved documents", 1, 5, 3)
question = st.text_input("Ask an academic question")

if st.button("Search and answer"):
    if not question.strip():
        st.warning("Please enter a question.")
    else:
        result = assistant.answer(question, top_k=top_k)
        st.subheader("Retrieved evidence")
        st.write(result["answer"])

        for source in result["sources"]:
            with st.expander(
                f"{source['source_id']} — {source['title']}"
            ):
                st.write(source["text"])
                st.write(f"Similarity score: {source['score']:.3f}")
'''

(project_dir / "app.py").write_text(
    textwrap.dedent(app_code).lstrip(),
    encoding="utf-8"
)

(project_dir / "requirements.txt").write_text(
    "streamlit\\nsentence-transformers\\nfaiss-cpu\\nnumpy\\n",
    encoding="utf-8"
)

(project_dir / ".gitignore").write_text(
    ".venv/\\n__pycache__/\\n*.pyc\\n.env\\n"
    ".streamlit/secrets.toml\\n",
    encoding="utf-8"
)

readme = """
# AI Academic Assistant using RAG

A reusable academic-support prototype using semantic retrieval,
FAISS vector search, source references, and Streamlit.

## Features
- JSONL academic FAQ ingestion
- Sentence Transformers embeddings
- FAISS top-k semantic retrieval
- Source-aware evidence presentation
- Streamlit interface
- Retrieval evaluation notebook

## Architecture
Question -> Embedding -> FAISS Retrieval -> Evidence + Source IDs

## Run locally
Install Python 3.10 or 3.11, then run:

```bash
pip install -r requirements.txt
streamlit run app.py

_IncompleteInputError: incomplete input (4184648374.py, line 157)

In [15]:
from pathlib import Path

project_dir = Path("/content/ai-academic-assistant-rag")
project_dir.mkdir(parents=True, exist_ok=True)
(project_dir / "data").mkdir(exist_ok=True)
(project_dir / "tests").mkdir(exist_ok=True)

print("Project directory ready:", project_dir)

Project directory ready: /content/ai-academic-assistant-rag


In [16]:
from pathlib import Path

project_dir = Path("/content/ai-academic-assistant-rag")

for path in sorted(project_dir.rglob("*")):
    print(path.relative_to(project_dir))

data
tests


In [17]:
from pathlib import Path

project_dir = Path("/content/ai-academic-assistant-rag")

required_files = [
    "data/academic_faqs.jsonl",
    "rag_engine.py",
    "app.py",
    "requirements.txt",
    "README.md",
    "tests/test_retrieval.py",
]

for filename in required_files:
    path = project_dir / filename
    status = "OK" if path.exists() and path.stat().st_size > 0 else "MISSING"
    print(f"{status}: {filename}")

MISSING: data/academic_faqs.jsonl
MISSING: rag_engine.py
MISSING: app.py
MISSING: requirements.txt
MISSING: README.md
MISSING: tests/test_retrieval.py


In [18]:

from pathlib import Path
import json

project_dir = Path("/content/ai-academic-assistant-rag")
(project_dir / "data").mkdir(parents=True, exist_ok=True)
(project_dir / "tests").mkdir(parents=True, exist_ok=True)

# 1. Sample knowledge base
docs = [
    {
        "source_id": "DOC001",
        "title": "Course Prerequisites",
        "text": "Students should complete the required prerequisite courses before enrolling in an advanced course. Check the official course catalog for details."
    },
    {
        "source_id": "DOC002",
        "title": "Course Withdrawal",
        "text": "Students who want to withdraw from a course should review the academic calendar and follow the institution's official withdrawal procedure."
    },
    {
        "source_id": "DOC003",
        "title": "Academic Advising",
        "text": "Academic advisors help students understand degree requirements, select courses, and plan their academic progress."
    },
    {
        "source_id": "DOC004",
        "title": "Examination Schedule",
        "text": "Students should check official institutional announcements for examination dates, room assignments, and schedule changes."
    },
    {
        "source_id": "DOC005",
        "title": "Student Data Privacy",
        "text": "Student records contain personal information. Access should be restricted to authorized users, and sensitive information should be protected."
    },
    {
        "source_id": "DOC006",
        "title": "Contacting an Academic Office",
        "text": "Students should contact the relevant academic office through its official communication channels for administrative questions."
    }
]

data_file = project_dir / "data" / "academic_faqs.jsonl"
with data_file.open("w", encoding="utf-8") as f:
    for doc in docs:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

# 2. Requirements
(project_dir / "requirements.txt").write_text(
    "sentence-transformers\nfaiss-cpu\nnumpy\npandas\nstreamlit\n",
    encoding="utf-8"
)

# 3. Git ignore
(project_dir / ".gitignore").write_text(
    "__pycache__/\n.venv/\n.env\n.ipynb_checkpoints/\n",
    encoding="utf-8"
)

# 4. Retrieval engine
rag_code = r'''
import json
from pathlib import Path

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


class AcademicRAG:
    def __init__(self, data_path="data/academic_faqs.jsonl"):
        self.data_path = Path(data_path)
        self.documents = [
            json.loads(line)
            for line in self.data_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]

        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

        texts = [
            doc["title"] + ". " + doc["text"]
            for doc in self.documents
        ]

        embeddings = self.model.encode(
            texts, normalize_embeddings=True
        )
        embeddings = np.asarray(embeddings, dtype="float32")

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

    def retrieve(self, question, top_k=3):
        query_vector = self.model.encode(
            [question], normalize_embeddings=True
        )
        query_vector = np.asarray(query_vector, dtype="float32")

        k = min(top_k, len(self.documents))
        scores, indices = self.index.search(query_vector, k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0:
                item = dict(self.documents[int(idx)])
                item["score"] = float(score)
                results.append(item)
        return results

    def answer(self, question, top_k=3):
        results = self.retrieve(question, top_k)
        if not results:
            return "No relevant source was found.", []

        evidence = "\n\n".join(
            f"[{item['source_id']}] {item['title']}: {item['text']}"
            for item in results
        )
        response = (
            "Retrieved evidence (not a generated answer):\n\n" + evidence
        )
        return response, results
'''
(project_dir / "rag_engine.py").write_text(rag_code.strip() + "\n", encoding="utf-8")

# 5. Streamlit interface
app_code = r'''
import streamlit as st
from rag_engine import AcademicRAG

st.set_page_config(page_title="Academic RAG Assistant", page_icon="📚")
st.title("AI Academic Assistant")
st.write("Ask a question and inspect the retrieved source documents.")

@st.cache_resource
def load_assistant():
    return AcademicRAG("data/academic_faqs.jsonl")

try:
    assistant = load_assistant()
    question = st.text_input("Your question")
    if question:
        answer, sources = assistant.answer(question)
        st.write(answer)
        st.subheader("Source details")
        for source in sources:
            st.write(
                f"{source['source_id']} — {source['title']} "
                f"(similarity: {source['score']:.3f})"
            )
except Exception as exc:
    st.error(f"Could not load the assistant: {exc}")
'''
(project_dir / "app.py").write_text(app_code.strip() + "\n", encoding="utf-8")

# 6. Basic retrieval test
test_code = r'''
import unittest
from rag_engine import AcademicRAG


class TestAcademicRAG(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.assistant = AcademicRAG("data/academic_faqs.jsonl")

    def test_retrieval_returns_results(self):
        results = self.assistant.retrieve("How do I protect student records?", top_k=3)
        self.assertGreater(len(results), 0)

    def test_results_have_source_ids(self):
        results = self.assistant.retrieve("Who can help with course planning?", top_k=3)
        self.assertTrue(all("source_id" in item for item in results))


if __name__ == "__main__":
    unittest.main()
'''
(project_dir / "tests" / "test_retrieval.py").write_text(test_code.strip() + "\n", encoding="utf-8")

# 7. README
readme = """
# AI Academic Assistant — Retrieval-Augmented Generation (RAG)

## Overview
A small educational knowledge-retrieval project demonstrating document ingestion,
semantic embeddings, FAISS vector search, source IDs, and basic retrieval tests.

## Architecture
JSONL documents -> SentenceTransformer embeddings -> FAISS similarity search
-> retrieved evidence with source IDs.

## Setup
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Example questions
- How do I protect student records?
- Who can help me plan my courses?
- Where can I check exam dates?

## Limitations
This prototype retrieves evidence; it does not generate answers with an LLM.
The sample documents are synthetic examples, not official institutional policies.
It does not yet implement fine-tuning, a knowledge graph, or PII redaction.

## Evaluation
Run the retrieval tests with:
```bash
python -m unittest discover -s tests -v
```
"""
(project_dir / "README.md").write_text(readme.strip() + "\n", encoding="utf-8")

print("Project files created!")
for p in sorted(project_dir.rglob("*")):
    if p.is_file():
        print("OK:", p.relative_to(project_dir))


Project files created!
OK: .gitignore
OK: README.md
OK: app.py
OK: data/academic_faqs.jsonl
OK: rag_engine.py
OK: requirements.txt
OK: tests/test_retrieval.py


In [19]:
%cd /content/ai-academic-assistant-rag
!python -m py_compile app.py rag_engine.py

/content/ai-academic-assistant-rag


In [20]:
%cd /content/ai-academic-assistant-rag

%pip install -q sentence-transformers faiss-cpu

/content/ai-academic-assistant-rag


In [21]:
!python -m unittest discover -s tests -v

Loading weights: 100% 103/103 [00:00<00:00, 9871.21it/s]
test_results_have_source_ids (test_retrieval.TestAcademicRAG.test_results_have_source_ids) ... ok
test_retrieval_returns_results (test_retrieval.TestAcademicRAG.test_retrieval_returns_results) ... ok

----------------------------------------------------------------------
Ran 2 tests in 2.186s

OK


In [22]:
import shutil

project_dir = "/content/ai-academic-assistant-rag"
zip_path = shutil.make_archive(
    "/content/ai-academic-assistant-rag",
    "zip",
    root_dir=project_dir
)

print("ZIP created:", zip_path)

ZIP created: /content/ai-academic-assistant-rag.zip


In [23]:
from google.colab import files

files.download("/content/ai-academic-assistant-rag.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
%pip install -q sentence-transformers faiss-cpu

In [25]:
%cd /content/ai-academic-assistant-rag
!python -m unittest discover -s tests -v

/content/ai-academic-assistant-rag
Loading weights: 100% 103/103 [00:00<00:00, 5032.66it/s]
test_results_have_source_ids (test_retrieval.TestAcademicRAG.test_results_have_source_ids) ... ok
test_retrieval_returns_results (test_retrieval.TestAcademicRAG.test_retrieval_returns_results) ... ok

----------------------------------------------------------------------
Ran 2 tests in 2.328s

OK


In [26]:
from rag_engine import AcademicRAG

assistant = AcademicRAG("data/academic_faqs.jsonl")

question = "How can I protect student records?"
results = assistant.retrieve(question, top_k=3)

for item in results:
    print("Source:", item["source_id"])
    print("Title:", item["title"])
    print("Score:", round(item["score"], 3))
    print("Evidence:", item["text"])
    print("-" * 50)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Source: DOC005
Title: Student Data Privacy
Score: 0.769
Evidence: Student records contain personal information. Access should be restricted to authorized users, and sensitive information should be protected.
--------------------------------------------------
Source: DOC006
Title: Contacting an Academic Office
Score: 0.396
Evidence: Students should contact the relevant academic office through its official communication channels for administrative questions.
--------------------------------------------------
Source: DOC002
Title: Course Withdrawal
Score: 0.257
Evidence: Students who want to withdraw from a course should review the academic calendar and follow the institution's official withdrawal procedure.
--------------------------------------------------
